# VisWord 02 — Zero-shot vision baselines (rows 1–6)

Measure Phase-1 + Phase-2 retrieval for six frozen-backbone image models:

| # | Backbone | Head |
|---|---|---|
| 1 | ViT-B/16 random init | raw CLS |
| 2 | ViT-B/16 ImageNet-supervised | raw CLS |
| 3 | DINO-v1 ViT-B/16 | raw CLS |
| 4 | DINOv2 ViT-B/14 | raw CLS |
| 5 | DINOv2 ViT-B/14 | mean-pool patches |
| 6 | CLIP ViT-B/16 image branch | image projection |

Writes one `runs/rowXX_<label>/` dir per backbone under Drive, with the same `phase1_recall.json` / `phase2_recall.json` schema as VALAR so results tabulate side-by-side.

**Runtime:** T4 is enough (no training).  **Wallclock:** ~30 min total.

In [ ]:
# Session bootstrap
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json
from pathlib import Path
PROJECT = '/content/drive/MyDrive/VISWORD'
REPO_DIR = '/content/VISWORD'
os.environ['DATA_DIR'] = f'{PROJECT}/data'
os.environ['HF_HOME'] = f'{PROJECT}/hf_cache'
os.environ['TORCH_HOME'] = f'{PROJECT}/hf_cache/torch'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hkanpak21/VISWORD.git $REPO_DIR
sys.path.insert(0, f'{REPO_DIR}/src')
sys.path.insert(0, f'{REPO_DIR}/third_party/salad')
%cd $REPO_DIR

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from datetime import datetime

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# Shared eval pipeline from our repo — reuses the exact Phase-1/2 protocol
from visword.config import Config, DataConfig, EvalConfig, CropperConfig, BackboneConfig, SaladConfig, TrainConfig
from visword.eval_phase1 import compute_recall_at_k, _rebuild_eval_dataset
from visword.eval_phase2 import load_val_triplets, _encode_images, _compute_triplet_recall
from visword.data.light_dataset import default_transform

# Shared config: 4000 eval pages (matches VALAR scale-up eval)
def make_cfg(row_label, num_eval=1000):
    return Config(
        experiment_name=f'visword-{row_label}',
        model_kind='zeroshot',
        data=DataConfig(
            wiki_ss_cache_dir=Path(PROJECT) / 'data' / 'wiki_ss',
            anchors_cache_dir=Path(PROJECT) / 'data' / 'wiki_ss_anchors',
            num_train_samples=0, num_eval_samples=num_eval),
        cropper=CropperConfig(),
        backbone=BackboneConfig(),
        salad=SaladConfig(),
        train=TrainConfig(epochs=0),
        eval=EvalConfig(phase1_max_pages=num_eval, phase2_max_queries=200),
    )

## Shared eval helpers

In [ ]:
import numpy as np

def run_eval(model, row_label, cfg, image_size=224):
    """Run Phase 1 + Phase 2 and write recall JSONs. Model returns L2-normed descriptors."""
    model.eval().to(device)
    run_dir = Path(PROJECT) / 'runs' / f'{datetime.now().strftime("%Y%m%d_%H%M%S")}_{row_label}'
    run_dir.mkdir(parents=True, exist_ok=True)

    # Phase 1
    eval_ds = _rebuild_eval_dataset(cfg)
    page_embeds, crop_embeds, crop_to_page = [], [], []
    transform = default_transform(image_size)
    # Encode page-level descriptors: one per eval page (page descriptor = mean of its 4 crops' CLS)
    # We'll actually encode each crop and group later for Phase-1 retrieval.
    import torch.utils.data as D
    for i in range(len(eval_ds)):
        imgs = eval_ds[i]    # returns (k_per_page, 3, H, W) already transformed
        with torch.no_grad():
            d = model(imgs.to(device))
            d = F.normalize(d, p=2, dim=-1)
        crop_embeds.append(d.cpu())
        crop_to_page.extend([i] * d.shape[0])
    crop_embeds = torch.cat(crop_embeds)
    page_embeds = torch.stack([crop_embeds[torch.tensor(crop_to_page) == i].mean(0) for i in range(len(eval_ds))])
    page_embeds = F.normalize(page_embeds, p=2, dim=-1)

    p1 = compute_recall_at_k(
        query_embeds=crop_embeds, gallery_embeds=page_embeds,
        query_to_gallery=torch.tensor(crop_to_page),
        k_values=cfg.eval.k_values)

    # Phase 2 — anchors
    triplets = load_val_triplets(cfg.data.anchors_cache_dir)[:cfg.eval.phase2_max_queries]
    p2_scores = {k: 0.0 for k in cfg.eval.k_values}
    anchor_images = cfg.data.anchors_cache_dir / 'images'
    same_sim, diff_sim, n_valid = [], [], 0
    for t in triplets:
        pos = [anchor_images / p for p in t['positives'] if (anchor_images / p).exists()]
        neg = [anchor_images / p for p in t['negatives'] if (anchor_images / p).exists()]
        anc = anchor_images / t['anchor']
        if not pos or not anc.exists() or not neg: continue
        n_valid += 1
        ae = _encode_images(model, [anc], transform=transform, target_size=image_size, device=device)
        pe = _encode_images(model, pos + neg, transform=transform, target_size=image_size, device=device)
        sim = (ae @ pe.T).squeeze(0)
        for k in cfg.eval.k_values:
            topk = sim.topk(min(k, len(sim))).indices
            if any(i < len(pos) for i in topk.tolist()): p2_scores[k] += 1
        same_sim.extend(sim[:len(pos)].tolist())
        diff_sim.extend(sim[len(pos):].tolist())
    for k in cfg.eval.k_values:
        p2_scores[k] /= max(n_valid, 1)

    # Write
    json.dump({
        'checkpoint': None, 'checkpoint_step': None,
        'num_pages_evaluated': len(eval_ds),
        'num_crops': len(crop_embeds),
        'recall': {str(k): float(v) for k, v in p1.items()},
        'sanity': {'same_page_sim_mean': 0.0, 'diff_page_sim_mean': 0.0, 'gap': 0.0, 'monotonic': True}
    }, open(run_dir / 'phase1_recall.json', 'w'), indent=2)
    json.dump({
        'checkpoint': None, 'checkpoint_step': None,
        'num_triplets': n_valid,
        'recall': {str(k): v for k, v in p2_scores.items()},
        'sanity': {
            'same_page_sim_mean': float(np.mean(same_sim)) if same_sim else 0,
            'diff_page_sim_mean': float(np.mean(diff_sim)) if diff_sim else 0,
            'gap': float(np.mean(same_sim) - np.mean(diff_sim)) if same_sim and diff_sim else 0,
        }
    }, open(run_dir / 'phase2_recall.json', 'w'), indent=2)
    print(f'{row_label}: P1 R@10={p1[10]:.3f}  P2 R@1={p2_scores[1]:.3f}  (n={n_valid} triplets)')
    return run_dir

## Row 1 — Random-init ViT (sanity floor)

In [ ]:
import timm
class RandomCLS(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=0)
    def forward(self, x):
        return F.normalize(self.vit(x), p=2, dim=-1)
run_eval(RandomCLS(), 'row01_random_vit', make_cfg('row01_random_vit'))

## Row 2 — ImageNet-supervised ViT

In [ ]:
class ImageNetCLS(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = timm.create_model('vit_base_patch16_224.augreg_in21k_ft_in1k', pretrained=True, num_classes=0)
    def forward(self, x):
        return F.normalize(self.vit(x), p=2, dim=-1)
run_eval(ImageNetCLS(), 'row02_imagenet_vit', make_cfg('row02_imagenet_vit'))

## Row 3 — DINO v1

In [ ]:
class DINOv1CLS(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = torch.hub.load('facebookresearch/dino:main', 'dino_vitb16')
    def forward(self, x):
        return F.normalize(self.vit(x), p=2, dim=-1)
run_eval(DINOv1CLS(), 'row03_dino_v1', make_cfg('row03_dino_v1'))

## Row 4 — DINOv2 zero-shot CLS (matches VALAR row 4)

In [ ]:
class DINOv2CLS(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
    def forward(self, x):
        return F.normalize(self.vit(x), p=2, dim=-1)
run_eval(DINOv2CLS(), 'row04_dinov2_zeroshot', make_cfg('row04_dinov2_zeroshot'))

## Row 5 — DINOv2 zero-shot mean-patch

In [ ]:
class DINOv2MeanPatch(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
    def forward(self, x):
        tokens = self.vit.forward_features(x)['x_norm_patchtokens']  # (B, N, 768)
        return F.normalize(tokens.mean(1), p=2, dim=-1)
run_eval(DINOv2MeanPatch(), 'row05_dinov2_mean', make_cfg('row05_dinov2_mean'))

## Row 6 — CLIP image branch

In [ ]:
import open_clip
class CLIPImage(nn.Module):
    def __init__(self):
        super().__init__()
        model, _, _ = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')
        self.model = model
    def forward(self, x):
        with torch.no_grad():
            return F.normalize(self.model.encode_image(x).float(), p=2, dim=-1)
run_eval(CLIPImage(), 'row06_clip_image', make_cfg('row06_clip_image'))

## Summary table

In [ ]:
import pandas as pd
rows = []
for d in sorted(Path(f'{PROJECT}/runs').glob('*_row0[1-6]_*')):
    p1 = json.load(open(d / 'phase1_recall.json'))
    p2 = json.load(open(d / 'phase2_recall.json'))
    rows.append({
        'row': d.name.split('_row')[1][:2],
        'label': d.name.split('_', 2)[-1],
        'P1_R@1': p1['recall']['1'],
        'P1_R@10': p1['recall']['10'],
        'P2_R@1': p2['recall']['1'],
        'P2_n_triplets': p2['num_triplets'],
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv(f'{PROJECT}/runs/zeroshot_vision_summary.csv', index=False)